In [ ]:
# %% Deep learning - Section 24.217
#    Predicting alternating sequences

# This code pertains a deep learning course provided by Mike X. Cohen on Udemy:
#   > https://www.udemy.com/course/deeplearning_x
# The "base" code in this repository is adapted (with very minor modifications)
# from code developed by the course instructor (Mike X. Cohen), while the
# "exercises" and the "code challenges" contain more original solutions and
# creative input from my side. If you are interested in DL (and if you are
# reading this statement, chances are that you are), go check out the course, it
# is singularly good.

In [1]:
# %% Libraries and modules
import numpy                  as np
import matplotlib.pyplot      as plt
import torch
import torch.nn               as nn
import seaborn                as sns
import copy
import torch.nn.functional    as F
import pandas                 as pd
import scipy.stats            as stats
import sklearn.metrics        as skm
import time
import sys
import imageio.v2
import torchvision
import torchvision.transforms as T
import torch.nn.utils         as utils
import random

from torch.utils.data                 import DataLoader,TensorDataset,Dataset,Subset
from sklearn.model_selection          import train_test_split
from google.colab                     import files
from torchsummary                     import summary
from scipy.stats                      import zscore
from sklearn.decomposition            import PCA
from scipy.signal                     import convolve2d
from torchsummary                     import summary
from matplotlib.gridspec              import GridSpec
from IPython                          import display
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')
plt.style.use('default')


In [ ]:
# %% Generate data

# Data
N    = 50
data = torch.zeros(N)

for i in range(N):
    data[i] = torch.rand(1) * (-1)**i

# Plot
phi = (1 + np.sqrt(5)) / 2
plt.figure(figsize=(phi*6,6))

plt.plot([-1,N+1],[0,0],'--',color=[.8,.8,.8])
plt.plot(data,'ks-',markerfacecolor='w')
plt.xlim([-1,N+1])
plt.title('Some data')

plt.savefig('figure1_alternating_sequences.png')
plt.show()
files.download('figure1_alternating_sequences.png')


In [79]:
# %% RNN model class

class RNN(nn.Module):
    def __init__(self,input_size,num_hidden,num_layers):
        super().__init__()

        # RNN layer(s) and output
        self.rnn = nn.RNN(input_size,num_hidden,num_layers)
        self.out = nn.Linear(num_hidden,1)

    def forward(self,x):

        # Pass through RNN layers (no explicit hidden initialisation)
        y,hidden = self.rnn(x)

        # Pass the RNN output through the fc output layer
        o = self.out(y)

        return o,hidden


In [ ]:
# %% Model's parameters

# Parameters
input_size =  1   # The data "channels"
num_hidden =  5   # Breadth of model (number of units in hidden layers)
num_layers =  1   # Depth of model (number of hidden layers)
seq_length =  9   # Number of datapoints used for learning in each segment
batch_size =  1   # (training code is hard-coded to organize data into batchsize=1)

# create an instance of the model and inspect
net = RNN(input_size,num_hidden,num_layers)

X   = torch.rand(seq_length,batch_size,input_size)
y,h = net(X)

# Note one output per sequence element (y.shape); generally, we take the final
# output to force a "many-to-one" design
print(X.shape)
print(y.shape)
print(h.shape)


In [ ]:
# %% Test the model on some data

# Data (transform into tensor with .view())
some_data = data[:seq_length].view(seq_length,1,1)
y = net(some_data)

# Grab final predicted value from the output (first element of tuple output of net)
final_value = y[0][-1]

# Loss (MSE is fine here even though binary CE is more appropriate)
loss_fun = nn.MSELoss()
loss_fun(final_value,data[seq_length].view(1,1))


In [ ]:
# %% Train model

# Epochs
num_epochs = 30

# New model and optimizer instance (SGD often used for standard RNNs)
net       = RNN(input_size,num_hidden,num_layers)
optimizer = torch.optim.SGD(net.parameters(),lr=.001)

# Preallocate losses and accuracy
losses        = np.zeros(num_epochs)
sign_accuracy = np.zeros(num_epochs)

# Loop
for epoch_i in range(num_epochs):

    # Loop over data segments (reset the hidden state on each epoch)
    seg_los      = []
    seg_acc      = []
    hidden_state = torch.zeros(num_layers,batch_size,num_hidden)

    for time_i in range(N-seq_length):

        # Grab data snippet (conceptually, x is size [1,9], y is size [1,1])
        X = data[time_i:time_i+seq_length].view(seq_length,1,1)
        y = data[time_i+seq_length].view(1,1)

        # Forward propagation and loss (compare final value of output)
        yHat,hidden_state = net(X)
        final_value       = yHat[-1]
        loss              = loss_fun(final_value,y)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Loss from current segment
        seg_los.append(loss.item())

        # Sign accuracy
        true_sign = np.sign(torch.squeeze(y).numpy())
        pred_sign = np.sign(torch.squeeze(final_value).detach().numpy())
        accuracy  = 100*(true_sign==pred_sign)
        seg_acc.append(accuracy)

    # Average losses from current epoch
    losses[epoch_i]       = np.mean(seg_los)
    sign_accuracy[epoch_i] = np.mean(seg_acc)

    msg = f'Finished epoch {epoch_i+1}/{num_epochs}'
    sys.stdout.write('\r' + msg)


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig,ax = plt.subplots(1,2,figsize=(1.5*phi*6,6))

ax[0].plot(losses,'s-')
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Loss')
ax[0].set_title('Model loss')

ax[1].plot(sign_accuracy,'m^-',markerfacecolor='g',markersize=10)
ax[1].set_xlabel('Epochs')
ax[1].set_ylabel('Accuracy')
ax[1].set_title(f'Sign accuracy (final accuracy: {sign_accuracy[-1]:.2f}%)')

plt.savefig('figure2_alternating_sequences.png')
plt.show()
files.download('figure2_alternating_sequences.png')


In [ ]:
# %% Test the model on the same data

# Initialize hidden state
h = np.zeros((N,num_hidden))

# Initialize predicted values (either zeros or nans)
yHat    = np.zeros(N)
yHat[:] = np.nan

# Loop over time segments
for time_i in range(N-seq_length):

    # Grab data snippet
    X = data[time_i:time_i+seq_length].view(seq_length,1,1)

    # Forward propagation and loss (also extract the hidden states)
    yy,hh = net(X)
    yHat[time_i+seq_length] = yy[-1]
    h[time_i+seq_length,:] = hh.detach()

# Compute sign-accuracy (only pick seq_length vals because the first n elements
# cannot be predicted)
true_sign     = np.sign(data.numpy())
pred_sign     = np.sign(yHat)
sign_accuracy = 100*np.mean(true_sign[seq_length:]==pred_sign[seq_length:])


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig,ax = plt.subplots(1,3,figsize=(1.5*phi*6,6))

ax[0].plot(data,'bs-',label='Actual data')
ax[0].plot(yHat,'ro-',label='Predicted')
ax[0].set_ylim([-1.1,1.1])
ax[0].set_title(f'Sign accuracy (final accuracy: {sign_accuracy:.2f}%)')
ax[0].legend()

ax[1].plot(data-yHat,'k^')
ax[1].set_ylim([-1.1,1.1])

ax[2].plot(data[seq_length:],yHat[seq_length:],'mo')
ax[2].set_xlabel('Real data')
ax[2].set_ylabel('Predicted data')
r = np.corrcoef(data[seq_length:],yHat[seq_length:])
ax[2].set_title(f"r={r[0,1]:.2f} (but Simpson's paradox!)")

plt.suptitle('Performance on training data',fontweight='bold',fontsize=20,y=1.1)
plt.tight_layout()

plt.savefig('figure3_alternating_sequences.png')
plt.show()
files.download('figure3_alternating_sequences.png')


In [ ]:
# %% Plotting

# Show the hidden "states" (units activations)
phi = (1 + np.sqrt(5)) / 2
plt.figure(figsize=(1.5*phi*6,6))

plt.plot(h,'s-')
plt.xlabel('Sequence index')
plt.ylabel('State value (a.u.)')
plt.title('Each line is a different hidden unit')

plt.savefig('figure4_alternating_sequences.png')
plt.show()
files.download('figure4_alternating_sequences.png')


In [ ]:
# %% Test on new data

# Create new data (with flipped sign!)
new_data = torch.zeros(N)
for i in range(N):
    new_data[i] = torch.rand(1) * (-1)**(i+1)

new_data = generate_pattern_sequence(pattern, N)

# Test the network (no learning here)
h    = np.zeros((N,num_hidden))
yHat = np.zeros(N)

for time_i in range(N-seq_length):

    # Grab data snippet
    X = new_data[time_i:time_i+seq_length].view(seq_length,1,1)

    # Forward propagation and loss
    yy,hh = net(X)
    yHat[time_i+seq_length] = yy[-1]
    h[time_i+seq_length,:] = hh.detach()

# Compute sign-accuracy
true_sign     = np.sign(new_data.numpy())
pred_sign     = np.sign(yHat)
sign_accuracy = 100*np.mean(true_sign[seq_length:]==pred_sign[seq_length:])


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig,ax = plt.subplots(1,3,figsize=(1.5*phi*6,6))

ax[0].plot(new_data,'bs-',label='Actual data')
ax[0].plot(yHat,'ro-',label='Predicted')
ax[0].set_ylim([-1.1,1.1])
ax[0].legend()
ax[0].set_title(f'Sign accuracy (final accuracy: {sign_accuracy:.2f}%)')

ax[1].plot(new_data-yHat,'k^')
ax[1].set_ylim([-1.1,1.1])
ax[1].set_title(f'Sign accuracy = {sign_accuracy:.2f}%')

ax[2].plot(new_data[seq_length:],yHat[seq_length:],'mo')
ax[2].set_xlabel('Real data')
ax[2].set_ylabel('Predicted data')
r = np.corrcoef(new_data[seq_length:],yHat[seq_length:])
ax[2].set_title(f"r={r[0,1]:.2f} (but Simpson's paradox!)")

plt.suptitle('Performance on unseen test data',fontweight='bold',fontsize=20,y=1.1)
plt.tight_layout()

plt.savefig('figure5_alternating_sequences.png')
plt.show()
files.download('figure5_alternating_sequences.png')


In [ ]:
# %% Plotting

# Show weights for the input to hidden layers
phi = (1 + np.sqrt(5)) / 2
plt.figure(figsize=(phi*5,5))

plt.bar(range(num_hidden),net.rnn.weight_ih_l0.detach().squeeze())
plt.ylabel('Weight value')
plt.xlabel('Hidden unit index')
plt.title('Input to hidden weights')

plt.savefig('figure6_alternating_sequences.png')
plt.show()
files.download('figure6_alternating_sequences.png')


In [ ]:
# %% Exercise 1
#    Is the model overfitting? One way to check is by setting the signs to be random instead of alternating. You can do
#    this by modifying the data-generation code to normal random numbers without forcing the sign. What is the predicted
#    accuracy level in this case?

# Much worse of course because the data sequence is effectively random; even
# with 300 epochs the accuracy is still fairy low, and of course the model does
# not generalise to new random data (I guess if the data are truly random, then
# learning is not even possible conceptually, one can at best overfit the
# current data, but there's no way to generalise to randomness)

# Generate new data
N    = 50
data = torch.zeros(N)

for i in range(N):
    data[i] = torch.randn(1)

phi = (1 + np.sqrt(5)) / 2
plt.figure(figsize=(phi*6,6))

plt.plot([-1,N+1],[0,0],'--',color=[.8,.8,.8])
plt.plot(data,'ks-',markerfacecolor='w')
plt.xlim([-1,N+1])
plt.title('Some data')

plt.savefig('figure12_alternating_sequences_extra1.png')
plt.show()
files.download('figure12_alternating_sequences_extra1.png')


In [ ]:
# %% Exercise 2
#    The hidden state is typically initialized to zeros. Is that really the best initialization? Weights are initialized
#    to random numbers. What happens if you initialize the hidden state to randn()? Run the model several times to get a
#    sense of the general trends. Now try initializing to all 100 (instead of zeros). Why are you getting these results?

# So the randn() initialisation works pretty well, and surprisingly the 100 init
# also produces okay results. I was expecting the randn init to work well, but
# frankly for the 100 initialisation I was expeting problems related to
# exploding/vanishing gradients, especially given that we have tanh() as
# nonlinearity. Maybe the dataset here is so simple that it doesn't really
# matter that much? If anything, losses decrease a bit more slowly than in the
# other cases I also don't fully understand why the most common init is 0
# instead of randn().

# Try these
hidden_state = torch.randn(num_layers,batch_size,num_hidden)
hidden_state = 100+torch.zeros(num_layers,batch_size,num_hidden)


In [101]:
# %% Exercise 3
#    The data problem here (predicting alternating signs) is pretty easy. Would this same model architecture perform as
#    well for more complicated sequences? As a start, modify the data-generating code to have the sequence ++- (thus,
#    two positive numbers and a negative number, then repeat that sign-pattern over and over again). Once you have this
#    code, you can test a variety of sign-sequencies, like ++--, --+, ++---, etc. Lots of fun to be had ;)

# Ups and downs

# New data function
def generate_pattern_sequence(pattern, N):
    data = torch.zeros(N)

    pattern_len = len(pattern)

    for i in range(N):
        symbol = pattern[i % pattern_len]

        if symbol == '+':
            data[i] = torch.rand(1)          # positive
        elif symbol == '-':
            data[i] = -torch.rand(1)         # negative
        else:
            raise ValueError(f"Invalid symbol '{symbol}' in pattern")

    return data


In [ ]:
# %% Exercise 3
#    Continue ...

# New data (try ++-, and ++--- here)
N       = 50
pattern = '++-'

data = generate_pattern_sequence(pattern, N)

phi = (1 + np.sqrt(5)) / 2
plt.figure(figsize=(phi*6,6))

plt.plot([-1,N+1],[0,0],'--',color=[.8,.8,.8])
plt.plot(data,'ks-',markerfacecolor='w')
plt.xlim([-1,N+1])
plt.title('Some data')

plt.savefig('figure27_alternating_sequences_extra3.png')
plt.show()
files.download('figure27_alternating_sequences_extra3.png')


In [ ]:
# %% Exercise 4
#    I set the sequence length to be 9. Do you think that's a good value here? Of course, this is a metaparameter that
#    you can pick, and the exact numerical value is a bit arbitrary. But some values are more appropriate than other
#    values, depending on the nature of the data. Based on what you know about our sequence data, what do you think about
#    the value of 9? What sequence lengths would be appropriate for the suggested sequences in question #3?

# Well the value should be picked, among other things, based on the "basic"
# frequency of the data, for example the '+-' sequence alternate every 2
# elements, but a sequence like '++-' has a cycle of 3 elements, so sequences
# with longer cycles should have a higher seq_length value. Trying here seq=15
# with a '++-' pattern, and it works much better than above.
